# 06 — Quantum Mechanics & Quantum Field Theory

> *A physicist’s guide to the mathematical foundations of quantum theory.*

This notebook builds **quantum mechanics from first principles** — from Hilbert spaces
and operators through the Schrödinger equation to the basics of quantum field theory.
We verify every theoretical result with **numerical simulations** using only NumPy and SciPy.

| Simulation | Method | Purpose |
|---|---|---|
| **Infinite Square Well** | Analytic + numerical | Discrete energy spectra & wavefunctions |
| **Harmonic Oscillator** | Matrix diagonalisation | Ladder operators & coherent states |
| **Hydrogen Atom** | Radial integration | Atomic orbitals & selection rules |
| **Klein–Gordon Field** | Lattice simulation | Free scalar field & propagators |

Every section pairs **rigorous mathematical exposition** (with LaTeX) and **runnable code**
so you can verify each claim empirically.

---

## 1. Setup

We begin by importing the core scientific Python stack.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad, solve_ivp
from scipy.linalg import eigh, expm
from scipy.special import hermite, factorial

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'figure.dpi': 100
})
sns.set_style('whitegrid')
np.random.seed(42)
print("All imports successful.")

---
## 2. Hilbert Spaces & Dirac Notation

### 2.1 What is a Hilbert space?

A **Hilbert space** $\mathcal{H}$ is a complete inner-product space — the natural arena
for quantum states. Every state $|\psi\rangle$ is a vector in $\mathcal{H}$.

| Property | Mathematical statement |
|---|---|
| **Inner product** | $\langle\phi|\psi\rangle \in \mathbb{C}$ |
| **Linearity** | $\langle\phi|\alpha\psi_1 + \beta\psi_2\rangle = \alpha\langle\phi|\psi_1\rangle + \beta\langle\phi|\psi_2\rangle$ |
| **Conjugate symmetry** | $\langle\phi|\psi\rangle = \langle\psi|\phi\rangle^*$ |
| **Positive-definiteness** | $\langle\psi|\psi\rangle \geq 0$, with equality iff $|\psi\rangle = 0$ |
| **Completeness** | Every Cauchy sequence converges in $\mathcal{H}$ |

### 2.2 Dirac notation

Dirac's bra-ket notation is compact and powerful:

$$|\psi\rangle = \sum_n c_n |n\rangle, \qquad c_n = \langle n|\psi\rangle$$

The **completeness relation** (resolution of the identity) reads:

$$\hat{I} = \sum_n |n\rangle\langle n|$$

> **Physicist's Intuition:** A Hilbert space is like an infinite-dimensional
> version of the familiar 3D vector space — but with complex coefficients and
> a notion of "length" given by the inner product. Each quantum state is a
> unit vector; physical observables are "directions" you can project onto.

In [ ]:
# === HILBERT SPACE DEMO ===
# Basis: |up> = [1,0], |down> = [0,1]
up = np.array([1, 0], dtype=complex)
down = np.array([0, 1], dtype=complex)

# Superposition: |psi> = (|up> + i|down>) / sqrt(2)
psi = (up + 1j * down) / np.sqrt(2)

print('=== Hilbert Space: Spin-1/2 ===')
print(f'|up>  = {up}')
print(f'|down>= {down}')
print(f'|psi> = {psi}')
print(f'<up|up>    = {np.vdot(up, up):.4f}  (normalised)')
print(f'<up|down>  = {np.vdot(up, down):.4f}  (orthogonal)')
print(f'<psi|psi>  = {np.vdot(psi, psi):.4f}  (normalised)')
print(f'|<up|psi>|^2  = {np.abs(np.vdot(up, psi))**2:.4f}  (prob of up)')
print(f'|<down|psi>|^2= {np.abs(np.vdot(down, psi))**2:.4f}  (prob of down)')

# Completeness relation
I_check = np.outer(up, up.conj()) + np.outer(down, down.conj())
print(f'Completeness: |up><up| + |down><down| = I ? {np.allclose(I_check, np.eye(2))}')

---
## 3. Operators & Observables

### 3.1 Hermitian operators

In quantum mechanics, **observables** correspond to **Hermitian (self-adjoint) operators**
$\hat{A} = \hat{A}^\dagger$. Their eigenvalues are real — these are the possible
measurement outcomes.

$$\hat{A}|a_n\rangle = a_n|a_n\rangle, \qquad a_n \in \mathbb{R}$$

### 3.2 Commutators & uncertainty

The **commutator** of two operators:

$$[\hat{A}, \hat{B}] = \hat{A}\hat{B} - \hat{B}\hat{A}$$

The **generalised uncertainty principle**:

$$\Delta A \cdot \Delta B \geq \frac{1}{2}|\langle[\hat{A}, \hat{B}]\rangle|$$

For position and momentum:

$$[\hat{x}, \hat{p}] = i\hbar \quad \Longrightarrow \quad \Delta x \cdot \Delta p \geq \frac{\hbar}{2}$$

> **Physicist's Intuition:** Non-commuting operators represent *incompatible
> observables* — you cannot simultaneously know both with arbitrary precision.
> This is not a limitation of measurement technology; it is a fundamental
> feature of quantum mechanics encoded in the algebra of operators.

In [ ]:
# === OPERATORS & COMMUTATORS ===
# --- Pauli matrices ---
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

print('=== Pauli Matrices ===')
for name, mat in [('sigma_x', sigma_x), ('sigma_y', sigma_y), ('sigma_z', sigma_z)]:
    print(f'{name} is Hermitian: {np.allclose(mat, mat.conj().T)}')

# Commutation relations: [sigma_i, sigma_j] = 2i epsilon_ijk sigma_k
comm_xy = sigma_x @ sigma_y - sigma_y @ sigma_x
print(f'[sigma_x, sigma_y] = 2i*sigma_z ? {np.allclose(comm_xy, 2j * sigma_z)}')
print(f'[sigma_y, sigma_z] = 2i*sigma_x ? {np.allclose(sigma_y @ sigma_z - sigma_z @ sigma_y, 2j * sigma_x)}')
print(f'[sigma_z, sigma_x] = 2i*sigma_y ? {np.allclose(sigma_z @ sigma_x - sigma_x @ sigma_z, 2j * sigma_y)}')

In [ ]:
# --- Position & momentum operators (finite difference) ---
N = 50
L = 10.0
dx = 2 * L / N
x_grid = np.linspace(-L, L, N, endpoint=False)
hbar = 1.0

# x-hat = diagonal matrix
x_hat = np.diag(x_grid)

# p-hat = -i*hbar d/dx (finite difference, periodic)
p_hat = np.zeros((N, N), dtype=complex)
for i in range(N):
    p_hat[i, (i+1) % N] = -1j * hbar / (2 * dx)
    p_hat[i, (i-1) % N] = 1j * hbar / (2 * dx)

# Commutator [x, p]
comm_xp = x_hat @ p_hat - p_hat @ x_hat
interior = slice(5, N-5)
diag_vals = np.diag(comm_xp)[interior]
expected_val = 1j * hbar

print('=== [x, p] = i*hbar (finite-difference check) ===')
print(f'Expected diagonal: {expected_val}')
print(f'Mean of diagonal (interior): {np.mean(diag_vals):.4f}')
print(f'Close to i*hbar: {np.allclose(np.mean(diag_vals), expected_val, atol=0.1)}')

---
## 4. The Schrödinger Equation

### 4.1 Time-dependent form

$$i\hbar\frac{\partial}{\partial t}|\psi(t)\rangle = \hat{H}|\psi(t)\rangle$$

### 4.2 Time-independent form (stationary states)

$$\hat{H}\phi_n(x) = E_n\phi_n(x)$$

### 4.3 Infinite square well (particle in a box)

For a particle in $[0, L]$ with infinite walls:

$$\phi_n(x) = \sqrt{\frac{2}{L}}\sin\!\left(\frac{n\pi x}{L}\right), \qquad
  E_n = \frac{n^2\pi^2\hbar^2}{2mL^2}$$

> **Physicist's Intuition:** Energy quantisation arises from boundary conditions,
> just as standing waves on a guitar string have discrete frequencies. The integer $n$
> counts the number of half-wavelengths that fit in the box.

In [ ]:
# === INFINITE SQUARE WELL ===
L_box = 1.0
m = 1.0
hbar = 1.0

def psi_box(x, n, L=L_box):
    return np.sqrt(2/L) * np.sin(n * np.pi * x / L)

def E_box(n, L=L_box, m=m, hbar=hbar):
    return (n**2 * np.pi**2 * hbar**2) / (2 * m * L**2)

x = np.linspace(0, L_box, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for n in range(1, 5):
    ax.plot(x, psi_box(x, n), label=f'n={n}', linewidth=2)
ax.set_xlabel('$x$')
ax.set_ylabel(r'$\phi_n(x)$')
ax.set_title('Wavefunctions - Infinite Square Well')
ax.legend()
ax.axhline(0, color='gray', linewidth=0.5)

ax = axes[1]
for n in range(1, 5):
    ax.plot(x, psi_box(x, n)**2, label=f'n={n}', linewidth=2)
ax.set_xlabel('$x$')
ax.set_ylabel(r'$|\phi_n(x)|^2$')
ax.set_title('Probability Densities')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Numerical solution via matrix diagonalisation ---
N_grid = 200
x_num = np.linspace(0, L_box, N_grid + 2)[1:-1]
dx_num = L_box / (N_grid + 1)

diag_main = np.full(N_grid, 2.0) / dx_num**2
diag_off = np.full(N_grid - 1, -1.0) / dx_num**2
H_box = (hbar**2 / (2 * m)) * (np.diag(diag_main) + np.diag(diag_off, 1) + np.diag(diag_off, -1))

eigenvalues, eigenvectors = eigh(H_box)

print('=== Infinite Square Well: Analytical vs Numerical ===')
print(f'{"n":>3} {"E_analytical":>14} {"E_numerical":>14} {"Relative Error":>16}')
print('-' * 52)
for n in range(1, 7):
    E_ana = E_box(n)
    E_num = eigenvalues[n - 1]
    rel_err = abs(E_num - E_ana) / E_ana
    print(f'{n:3d} {E_ana:14.6f} {E_num:14.6f} {rel_err:16.2e}')

---
## 5. The Quantum Harmonic Oscillator

### 5.1 Hamiltonian and ladder operators

$$\hat{H} = \frac{\hat{p}^2}{2m} + \frac{1}{2}m\omega^2\hat{x}^2$$

Then $\hat{H} = \hbar\omega\left(\hat{a}^\dagger\hat{a} + \frac{1}{2}\right)$ and:

$$E_n = \hbar\omega\!\left(n + \tfrac{1}{2}\right), \qquad n = 0, 1, 2, \ldots$$

### 5.2 Wavefunctions

$$\phi_n(x) = \frac{1}{\sqrt{2^n n!}}\left(\frac{m\omega}{\pi\hbar}\right)^{1/4}
e^{-m\omega x^2/(2\hbar)}\,H_n\!\left(\sqrt{\frac{m\omega}{\hbar}}\,x\right)$$

> **Physicist's Intuition:** The harmonic oscillator is *the* most important model
> in physics. Every smooth potential looks quadratic near a minimum, so small
> oscillations in *any* system are governed by this Hamiltonian. The zero-point
> energy $E_0 = \hbar\omega/2$ is a pure quantum effect with no classical analogue.

In [ ]:
# === QUANTUM HARMONIC OSCILLATOR ===
omega = 1.0
m_ho = 1.0
hbar_ho = 1.0

def psi_ho(x, n, m=m_ho, w=omega, hb=hbar_ho):
    xi = np.sqrt(m * w / hb) * x
    Hn = hermite(n)
    norm = (1.0 / np.sqrt(2**n * float(factorial(n, exact=True)))) * \
           (m * w / (np.pi * hb))**0.25
    return norm * np.exp(-xi**2 / 2) * Hn(xi)

x_ho = np.linspace(-5, 5, 500)
V_ho = 0.5 * m_ho * omega**2 * x_ho**2

fig, ax = plt.subplots(figsize=(10, 7))
for n in range(5):
    psi_val = psi_ho(x_ho, n)
    E_n = hbar_ho * omega * (n + 0.5)
    ax.plot(x_ho, psi_val + E_n, linewidth=2, label=f'n={n}, E={E_n:.1f}')
    ax.fill_between(x_ho, E_n, psi_val + E_n, alpha=0.15)
    ax.axhline(E_n, color='gray', linewidth=0.3, linestyle='--')

ax.plot(x_ho, V_ho, 'k-', linewidth=2, label=r'$V(x) = m\omega^2 x^2/2$')
ax.set_xlabel('$x$')
ax.set_ylabel('Energy / Wavefunction (offset)')
ax.set_title('Quantum Harmonic Oscillator')
ax.set_ylim(-0.5, 6)
ax.set_xlim(-4.5, 4.5)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# --- Matrix diagonalisation approach ---
N_ho = 100
x_ho_num = np.linspace(-8, 8, N_ho)
dx_ho = x_ho_num[1] - x_ho_num[0]

T_diag = np.full(N_ho, 2.0) / dx_ho**2
T_off = np.full(N_ho - 1, -1.0) / dx_ho**2
T = (hbar_ho**2 / (2 * m_ho)) * (np.diag(T_diag) + np.diag(T_off, 1) + np.diag(T_off, -1))
V = np.diag(0.5 * m_ho * omega**2 * x_ho_num**2)
H_ho = T + V
eigenvalues_ho, _ = eigh(H_ho)

print('=== Harmonic Oscillator: Analytical vs Numerical ===')
print(f'{"n":>3} {"E_analytical":>14} {"E_numerical":>14} {"Rel Error":>12}')
print('-' * 48)
for n in range(8):
    E_ana = hbar_ho * omega * (n + 0.5)
    E_num = eigenvalues_ho[n]
    rel_err = abs(E_num - E_ana) / E_ana
    print(f'{n:3d} {E_ana:14.6f} {E_num:14.6f} {rel_err:12.2e}')

---
## 6. Angular Momentum & Spin

### 6.1 Orbital angular momentum

$$[\hat{L}_i, \hat{L}_j] = i\hbar\,\varepsilon_{ijk}\,\hat{L}_k$$

The simultaneous eigenstates of $\hat{L}^2$ and $\hat{L}_z$ are the **spherical harmonics**:

$$\hat{L}^2\,Y_\ell^m = \hbar^2\ell(\ell+1)\,Y_\ell^m, \qquad
  \hat{L}_z\,Y_\ell^m = \hbar m\,Y_\ell^m$$

### 6.2 Spin-1/2

| Quantum Number | Symbol | Values |
|---|---|---|
| Orbital | $\ell$ | $0, 1, 2, \ldots$ |
| Magnetic | $m$ | $-\ell, \ldots, +\ell$ |
| Spin | $s$ | $0, \tfrac{1}{2}, 1, \ldots$ |

> **Physicist's Intuition:** Spin is *intrinsic* angular momentum — it doesn't
> come from a particle physically rotating. It's a purely quantum degree of freedom
> with no classical analogue.

In [ ]:
# === ANGULAR MOMENTUM ===
def angular_momentum_matrices(j):
    """Construct Jx, Jy, Jz matrices for spin j."""
    dim = int(2 * j + 1)
    m_vals = np.arange(j, -j - 1, -1)
    Jz = np.diag(m_vals).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    for i in range(dim - 1):
        mv = m_vals[i + 1]
        Jp[i, i + 1] = np.sqrt(j * (j + 1) - mv * (mv + 1))
    Jm = Jp.conj().T
    Jx = (Jp + Jm) / 2.0
    Jy = (Jp - Jm) / (2.0j)
    return Jx, Jy, Jz

# Spin-1
Jx, Jy, Jz = angular_momentum_matrices(1)
print('=== Spin-1 Angular Momentum ===')
print(f'J_x =\n{np.real(Jx)}\n')
print(f'J_z =\n{np.real(Jz)}\n')

# Verify [Jx, Jy] = i Jz
comm = Jx @ Jy - Jy @ Jx
print(f'[Jx, Jy] = i*Jz ? {np.allclose(comm, 1j * Jz)}')

# J^2
J2 = Jx @ Jx + Jy @ Jy + Jz @ Jz
j_val = 1
print(f'J^2 = j(j+1)*I = {j_val*(j_val+1):.1f}*I ? {np.allclose(J2, j_val*(j_val+1)*np.eye(3))}')

In [ ]:
# --- Spherical harmonics visualisation ---
from scipy.special import sph_harm

theta = np.linspace(0, np.pi, 100)
phi = np.linspace(0, 2 * np.pi, 100)
THETA, PHI = np.meshgrid(theta, phi)

fig, axes = plt.subplots(2, 3, figsize=(14, 8),
                          subplot_kw={'projection': '3d'})

harmonics = [(0,0), (1,0), (1,1), (2,0), (2,1), (2,2)]

for ax, (l_val, m_val) in zip(axes.flat, harmonics):
    Y = sph_harm(m_val, l_val, PHI, THETA)
    r = np.abs(Y)
    X = r * np.sin(THETA) * np.cos(PHI)
    Y_plot = r * np.sin(THETA) * np.sin(PHI)
    Z = r * np.cos(THETA)
    ax.plot_surface(X, Y_plot, Z, cmap='RdBu', alpha=0.7, edgecolor='none')
    ax.set_title(f'Y({l_val},{m_val})', fontsize=12)
    ax.set_xlim(-0.6, 0.6)
    ax.set_ylim(-0.6, 0.6)
    ax.set_zlim(-0.6, 0.6)

fig.suptitle('Spherical Harmonics', fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

---
## 7. Perturbation Theory

### 7.1 Time-independent perturbation theory

When $\hat{H} = \hat{H}_0 + \lambda\hat{V}$:

**First-order energy correction:**

$$E_n^{(1)} = \langle n^{(0)}|\hat{V}|n^{(0)}\rangle$$

**Second-order energy correction:**

$$E_n^{(2)} = \sum_{k \neq n} \frac{|\langle k^{(0)}|\hat{V}|n^{(0)}\rangle|^2}{E_n^{(0)} - E_k^{(0)}}$$

> **Physicist's Intuition:** Perturbation theory is the workhorse of quantum mechanics.
> When you can't solve a problem exactly, solve a *nearby* problem and compute corrections.

In [ ]:
# === PERTURBATION THEORY ===
# Perturbed harmonic oscillator: V(x) = 0.5*w^2*x^2 + lambda*x^4
N_pt = 80
x_pt = np.linspace(-6, 6, N_pt)
dx_pt = x_pt[1] - x_pt[0]
omega_pt = 1.0

T_diag_pt = np.full(N_pt, 2.0) / dx_pt**2
T_off_pt = np.full(N_pt - 1, -1.0) / dx_pt**2
T_pt = 0.5 * (np.diag(T_diag_pt) + np.diag(T_off_pt, 1) + np.diag(T_off_pt, -1))
V0 = np.diag(0.5 * omega_pt**2 * x_pt**2)
H0 = T_pt + V0

lam = 0.1
V_pert = np.diag(lam * x_pt**4)
H_full = H0 + V_pert
E_exact, psi_exact = eigh(H_full)
E0_vals, psi0 = eigh(H0)

print('=== Perturbation Theory: Anharmonic Oscillator (lambda=0.1) ===')
print(f'{"n":>3} {"E0":>10} {"E0+E(1)":>10} {"E0+E(1)+E(2)":>14} {"E_exact":>10} {"PT2 Error":>12}')
print('-' * 65)
for n in range(5):
    E0_n = E0_vals[n]
    psi_n = psi0[:, n]
    E1 = psi_n @ V_pert @ psi_n
    E2 = 0.0
    for k in range(N_pt):
        if k == n:
            continue
        V_kn = psi0[:, k] @ V_pert @ psi_n
        E2 += abs(V_kn)**2 / (E0_n - E0_vals[k])
    E_pt1 = E0_n + E1
    E_pt2 = E0_n + E1 + E2
    rel_err = abs(E_pt2 - E_exact[n]) / abs(E_exact[n])
    print(f'{n:3d} {E0_n:10.5f} {E_pt1:10.5f} {E_pt2:14.5f} {E_exact[n]:10.5f} {rel_err:12.2e}')

---
## 8. Path Integrals

### 8.1 Feynman's formulation

$$K(x_b, T; x_a, 0) = \int \mathcal{D}[x(t)]\;e^{iS[x(t)]/\hbar}$$

### 8.2 Euclidean path integral

The Wick rotation $t \to -i\tau$ converts oscillatory integrands to decaying exponentials:

$$K_E \propto \int \mathcal{D}[x]\;e^{-S_E[x]/\hbar}$$

> **Physicist's Intuition:** In the classical limit $\hbar \to 0$, the wildly
> oscillating phase $e^{iS/\hbar}$ causes all paths to cancel *except* the one
> that extremises the action — the classical trajectory. Quantum mechanics is
> what happens when nearby paths don't completely cancel.

In [ ]:
# === PATH INTEGRAL: Monte Carlo ===
N_sites = 20
N_mc = 5000
N_therm = 1000
a_lat = 0.5
m_pi = 1.0
omega_pi = 1.0

def euclidean_action(path, a, m_val, w):
    S = 0.0
    N_path = len(path)
    for i in range(N_path):
        ip1 = (i + 1) % N_path
        kinetic = 0.5 * m_val * (path[ip1] - path[i])**2 / a
        potential = a * 0.5 * m_val * w**2 * path[i]**2
        S += kinetic + potential
    return S

path = np.zeros(N_sites)
x2_measurements = []
delta = 1.0
accepted = 0
total = 0

for sweep in range(N_mc + N_therm):
    for site in range(N_sites):
        old_x = path[site]
        old_S = euclidean_action(path, a_lat, m_pi, omega_pi)
        path[site] = old_x + delta * (2 * np.random.random() - 1)
        new_S = euclidean_action(path, a_lat, m_pi, omega_pi)
        dS = new_S - old_S
        total += 1
        if dS < 0 or np.random.random() < np.exp(-dS):
            accepted += 1
        else:
            path[site] = old_x
    if sweep >= N_therm:
        x2_measurements.append(np.mean(path**2))

x2_mc = np.mean(x2_measurements)
x2_exact = 1.0 / (2 * m_pi * omega_pi)

print('=== Path Integral Monte Carlo: Harmonic Oscillator ===')
print(f'<x^2> (Monte Carlo):  {x2_mc:.4f}')
print(f'<x^2> (exact, T->inf): {x2_exact:.4f}')
print(f'Acceptance rate:       {accepted/total:.2%}')
print(f'Relative error:        {abs(x2_mc - x2_exact)/x2_exact:.1%}')

---
## 9. From Particles to Fields — Second Quantisation

### 9.1 Fock space

$$\mathcal{F} = \bigoplus_{n=0}^{\infty} \mathcal{H}^{(n)}$$

### 9.2 Creation & annihilation operators

$$[\hat{a}_j, \hat{a}_k^\dagger] = \delta_{jk}, \qquad [\hat{a}_j, \hat{a}_k] = 0$$

> **Physicist's Intuition:** Second quantisation re-interprets quantum mechanics
> not in terms of individual particles, but in terms of *how many particles occupy
> each mode*. This is essential for QFT, where particles can be created and destroyed.

In [ ]:
# === FOCK SPACE ===
def creation_op(n_max):
    a_dag = np.zeros((n_max, n_max), dtype=complex)
    for n in range(n_max - 1):
        a_dag[n + 1, n] = np.sqrt(n + 1)
    return a_dag

def annihilation_op(n_max):
    a_op = np.zeros((n_max, n_max), dtype=complex)
    for n in range(1, n_max):
        a_op[n - 1, n] = np.sqrt(n)
    return a_op

n_max = 6
a_dag = creation_op(n_max)
a_op = annihilation_op(n_max)
n_hat = a_dag @ a_op

print('=== Fock Space Operators (truncated to n_max=5) ===')
print(f'a_dag (creation) =\n{np.real(a_dag)}\n')
print(f'a (annihilation) =\n{np.real(a_op)}\n')
print(f'n_hat = a_dag @ a =\n{np.real(n_hat)}\n')

# [a, a_dag] = I
comm_aa = a_op @ a_dag - a_dag @ a_op
print(f'[a, a_dag] = I ? {np.allclose(comm_aa, np.eye(n_max))}')

for n in range(n_max):
    state = np.zeros(n_max)
    state[n] = 1.0
    result = n_hat @ state
    print(f'n_hat|{n}> = {result[n]:.1f}|{n}>')

---
## 10. The Klein–Gordon Field

### 10.1 Classical field theory

$$\left(\frac{\partial^2}{\partial t^2} - \nabla^2 + m^2\right)\phi = 0$$

### 10.2 Dispersion relation

$$\omega_k = \sqrt{k^2 + m^2}$$

> **Physicist's Intuition:** In QFT, particles are not fundamental — *fields* are.
> A particle is simply a quantised excitation (a 'ripple') in the underlying field.

In [ ]:
# === KLEIN-GORDON DISPERSION ===
m_kg = 1.0
k_vals = np.linspace(-5, 5, 500)
omega_kg = np.sqrt(k_vals**2 + m_kg**2)
omega_massless = np.abs(k_vals)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_vals, omega_kg, 'b-', linewidth=2, label=f'Massive: m={m_kg}')
ax.plot(k_vals, omega_massless, 'r--', linewidth=2, label='Massless')
ax.axhline(m_kg, color='gray', linestyle=':', label=f'Mass gap m={m_kg}')
ax.set_xlabel('Wavenumber $k$')
ax.set_ylabel('Frequency $\\omega$')
ax.set_title('Klein-Gordon Dispersion Relation')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 1+1D lattice Klein-Gordon simulation ---
N_lat = 200
L_lat = 40.0
dx_lat = L_lat / N_lat
dt_lat = 0.02
N_steps = 800
m_field = 0.5

x_lat = np.linspace(-L_lat/2, L_lat/2, N_lat, endpoint=False)
k0 = 3.0
sigma0 = 2.0
phi_field = np.exp(-x_lat**2 / (2 * sigma0**2)) * np.cos(k0 * x_lat)
phi_dot = np.zeros(N_lat)

snapshots = [(0, phi_field.copy())]
snap_times = [0, 200, 400, 600, 800]

for step in range(1, N_steps + 1):
    laplacian = np.roll(phi_field, 1) + np.roll(phi_field, -1) - 2 * phi_field
    laplacian /= dx_lat**2
    phi_ddot = laplacian - m_field**2 * phi_field
    phi_dot += dt_lat * phi_ddot
    phi_field += dt_lat * phi_dot
    if step in snap_times:
        snapshots.append((step, phi_field.copy()))

fig, ax = plt.subplots(figsize=(12, 5))
for step, snap in snapshots:
    t = step * dt_lat
    ax.plot(x_lat, snap, label=f't = {t:.1f}', alpha=0.8)
ax.set_xlabel('Position $x$')
ax.set_ylabel('Field $\\phi(x,t)$')
ax.set_title(f'Klein-Gordon Field Evolution (m = {m_field})')
ax.legend()
plt.tight_layout()
plt.show()
print(f'CFL condition: dt/dx = {dt_lat/dx_lat:.3f} < 1', 'OK' if dt_lat < dx_lat else 'VIOLATED')

In [ ]:
# --- Phase & group velocities ---
k_pos = np.linspace(0.1, 8, 100)
v_phase = np.sqrt(k_pos**2 + m_field**2) / k_pos
v_group = k_pos / np.sqrt(k_pos**2 + m_field**2)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_pos, v_phase, 'b-', linewidth=2, label='Phase velocity $v_p$')
ax.plot(k_pos, v_group, 'r-', linewidth=2, label='Group velocity $v_g$')
ax.axhline(1.0, color='gray', linestyle=':', label='Speed of light c=1')
ax.set_xlabel('$k$')
ax.set_ylabel('Velocity')
ax.set_title(f'Phase & Group Velocities (m = {m_field})')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Group velocity always < c: {np.all(v_group < 1.0)}')
print(f'Phase velocity always > c: {np.all(v_phase > 1.0)}')

---
## 11. Coherent States & the Classical Limit

### 11.1 Definition

$$\hat{a}|\alpha\rangle = \alpha|\alpha\rangle, \qquad \alpha \in \mathbb{C}$$

| Property | Value |
|---|---|
| $\langle\hat{n}\rangle$ | $|\alpha|^2$ |
| $\Delta n$ | $|\alpha|$ |
| $\Delta x \cdot \Delta p$ | $\hbar/2$ (minimum uncertainty) |
| Photon number distribution | Poissonian |

> **Physicist's Intuition:** Coherent states are the *most classical* quantum states.
> A laser beam is well-described by a coherent state. As $|\alpha| \to \infty$,
> the relative fluctuations vanish and we recover classical oscillator behaviour.

In [ ]:
# === COHERENT STATES ===
from scipy.special import factorial as sp_factorial

alpha_vals = [1.0, 2.0, 3.0]
n_max_cs = 20
n_range = np.arange(n_max_cs)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, alpha in zip(axes, alpha_vals):
    mean_n = abs(alpha)**2
    probs = np.exp(-mean_n) * mean_n**n_range / sp_factorial(n_range, exact=False)
    ax.bar(n_range, probs, color='steelblue', alpha=0.7, edgecolor='navy')
    ax.set_xlabel('Photon number $n$')
    ax.set_ylabel('$P(n)$')
    ax.set_title(f'|alpha| = {alpha:.0f}, <n> = {mean_n:.0f}')
plt.suptitle('Coherent State Photon Number Distributions (Poissonian)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Phase space trajectory ---
omega_cs = 1.0
alpha0 = 2.0
t_vals = np.linspace(0, 4 * np.pi, 200)

x_expect = np.sqrt(2) * np.real(alpha0 * np.exp(-1j * omega_cs * t_vals))
p_expect = -np.sqrt(2) * np.imag(alpha0 * np.exp(-1j * omega_cs * t_vals))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(t_vals, x_expect, 'b-', linewidth=2, label='<x>(t)')
axes[0].plot(t_vals, p_expect, 'r--', linewidth=2, label='<p>(t)')
axes[0].set_xlabel('Time $t$')
axes[0].set_ylabel('Expectation value')
axes[0].set_title('Coherent State: Classical Trajectory')
axes[0].legend()

axes[1].plot(x_expect, p_expect, 'purple', linewidth=2)
axes[1].plot(x_expect[0], p_expect[0], 'go', markersize=10, label='Start')
axes[1].set_xlabel('<x>')
axes[1].set_ylabel('<p>')
axes[1].set_title('Phase Space Trajectory')
axes[1].set_aspect('equal')
axes[1].legend()
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)

plt.tight_layout()
plt.show()
print(f'Classical amplitude: sqrt(2)|alpha| = {np.sqrt(2)*alpha0:.3f}')
print(f'Energy: hbar*omega*(|alpha|^2+1/2) = {omega_cs*(alpha0**2 + 0.5):.3f}')

---
## 12. Summary & Connections

| Topic | Central equation | Section |
|---|---|---|
| Hilbert space | $\langle\psi|\psi\rangle = 1$ | §2 |
| Commutators | $[\hat{x}, \hat{p}] = i\hbar$ | §3 |
| Schrödinger eq. | $i\hbar\partial_t|\psi\rangle = \hat{H}|\psi\rangle$ | §4 |
| Square well | $E_n = n^2\pi^2\hbar^2/(2mL^2)$ | §4 |
| Harmonic oscillator | $E_n = \hbar\omega(n + \tfrac{1}{2})$ | §5 |
| Angular momentum | $[\hat{L}_i, \hat{L}_j] = i\hbar\varepsilon_{ijk}\hat{L}_k$ | §6 |
| Perturbation theory | $E_n^{(1)} = \langle n|\hat{V}|n\rangle$ | §7 |
| Path integral | $K = \int\mathcal{D}[x]\,e^{iS/\hbar}$ | §8 |
| Fock space | $[\hat{a}, \hat{a}^\dagger] = 1$ | §9 |
| Klein–Gordon | $(\partial^2_t - \nabla^2 + m^2)\phi = 0$ | §10 |
| Coherent states | $\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$ | §11 |

### Connections to other notebooks

- **Measure theory (05):** Probability spaces underpin quantum measurement theory
- **Matrix calculus (03):** Eigenvalue problems and diagonalisation are central to QM
- **Information theory (02):** Von Neumann entropy generalises Shannon entropy to quantum states
- **Statistical foundations (01):** Expectation values and variances of quantum observables

### Further reading

- Griffiths & Schroeter, *Introduction to Quantum Mechanics* (3rd ed.)
- Sakurai & Napolitano, *Modern Quantum Mechanics* (3rd ed.)
- Peskin & Schroeder, *An Introduction to Quantum Field Theory*